# Практическая работа № 4. Численное решение уравнений и систем

## Цель
Освоить методы численного решения нелинейных уравнений, систем линейных и нелинейных алгебраических уравнений.

## Теоретический минимум
- **Нелинейные уравнения**: Методы бисекции, Ньютона, простых итераций и др. для f(x) = 0.
- **Линейные системы**: Метод Гаусса, итерационные методы.
- **Нелинейные системы**: Обобщение методов Ньютона и итераций на системы.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Решение нелинейных уравнений

In [3]:
class NonlinearEquationSolver:
    def __init__(self, f, df=None, ddf=None):
        self.f = f
        self.df = df
        self.ddf = ddf

    def localize_roots(self, a, b, n=100):
        # Локализация корней: поиск интервалов, где f меняет знак
        roots = []
        x_vals = np.linspace(a, b, n)
        for i in range(len(x_vals) - 1):
            if self.f(x_vals[i]) * self.f(x_vals[i+1]) < 0:
                roots.append((x_vals[i], x_vals[i+1]))
        return roots

    def bisection(self, a, b, tol=1e-6, max_iter=100):
        # Метод бисекции
        if self.f(a) * self.f(b) >= 0:
            raise ValueError("f(a) and f(b) must have opposite signs")
        for _ in range(max_iter):
            c = (a + b) / 2
            if abs(self.f(c)) < tol:
                return c
            if self.f(a) * self.f(c) < 0:
                b = c
            else:
                a = c
        return (a + b) / 2

    def fixed_point_iteration(self, g, x0, tol=1e-6, max_iter=100):
        # Метод простых итераций: x = g(x)
        x = x0
        for _ in range(max_iter):
            x_new = g(x)
            if abs(x_new - x) < tol:
                return x_new
            x = x_new
        return x

    def newton(self, x0, tol=1e-6, max_iter=100):
        # Метод Ньютона
        if self.df is None:
            raise ValueError("Derivative required for Newton method")
        x = x0
        for _ in range(max_iter):
            fx = self.f(x)
            dfx = self.df(x)
            if abs(dfx) < 1e-10:
                raise ValueError("Derivative too small")
            x_new = x - fx / dfx
            if abs(x_new - x) < tol:
                return x_new
            x = x_new
        return x

    def modified_newton(self, x0, tol=1e-6, max_iter=100):
        # Модифицированный метод Ньютона (с постоянной производной)
        if self.df is None:
            raise ValueError("Derivative required")
        dfx0 = self.df(x0)
        x = x0
        for _ in range(max_iter):
            fx = self.f(x)
            x_new = x - fx / dfx0
            if abs(x_new - x) < tol:
                return x_new
            x = x_new
        return x

    def muller(self, x0, x1, x2, tol=1e-6, max_iter=100):
        # Метод Мюллера
        h1 = x1 - x0
        h2 = x2 - x1
        d1 = (self.f(x1) - self.f(x0)) / h1
        d2 = (self.f(x2) - self.f(x1)) / h2
        d = (d2 - d1) / (h2 + h1)
        i = 0
        while i < max_iter:
            b = d2 + h2 * d
            D = np.sqrt(b**2 - 4 * self.f(x2) * d)
            if abs(b - D) < abs(b + D):
                E = b + D
            else:
                E = b - D
            h = -2 * self.f(x2) / E
            x3 = x2 + h
            if abs(h) < tol:
                return x3
            x0, x1, x2 = x1, x2, x3
            h1 = x1 - x0
            h2 = x2 - x1
            d1 = (self.f(x1) - self.f(x0)) / h1
            d2 = (self.f(x2) - self.f(x1)) / h2
            d = (d2 - d1) / (h2 + h1)
            i += 1
        return x3

    def secant(self, x0, x1, tol=1e-6, max_iter=100):
        # Метод хорд (секант)
        for _ in range(max_iter):
            if abs(self.f(x1) - self.f(x0)) < 1e-10:
                raise ValueError("Division by zero")
            x2 = x1 - self.f(x1) * (x1 - x0) / (self.f(x1) - self.f(x0))
            if abs(x2 - x1) < tol:
                return x2
            x0, x1 = x1, x2
        return x2

    def false_position(self, a, b, tol=1e-6, max_iter=100):
        # Метод ложного положения
        if self.f(a) * self.f(b) >= 0:
            raise ValueError("f(a) and f(b) must have opposite signs")
        for _ in range(max_iter):
            c = (a * self.f(b) - b * self.f(a)) / (self.f(b) - self.f(a))
            if abs(self.f(c)) < tol:
                return c
            if self.f(a) * self.f(c) < 0:
                b = c
            else:
                a = c
        return c

    def steffensen(self, x0, tol=1e-6, max_iter=100):
        # Метод Стэффенсена
        x = x0
        for _ in range(max_iter):
            y = self.f(x)
            z = self.f(x + y)
            if abs(z - 2*y) < 1e-10:
                raise ValueError("Division by zero")
            x_new = x - (y**2) / (z - 2*y)
            if abs(x_new - x) < tol:
                return x_new
            x = x_new
        return x

In [4]:
# Пример: f(x) = x^3 - x - 2
def f(x):
    return x**3 - x - 2

def df(x):
    return 3*x**2 - 1

solver = NonlinearEquationSolver(f, df)

# Локализация
roots = solver.localize_roots(1, 2)
print("Локализованные корни:", roots)

# Бисекция
root_bisect = solver.bisection(1, 2)
print("Корень бисекцией:", root_bisect)

# Ньютон
root_newton = solver.newton(1.5)
print("Корень Ньютона:", root_newton)

# Хорд
root_secant = solver.secant(1, 2)
print("Корень хорд:", root_secant)

Локализованные корни: [(np.float64(1.5151515151515151), np.float64(1.5252525252525253))]
Корень бисекцией: 1.5213797092437744
Корень Ньютона: 1.5213797068045751
Корень хорд: 1.5213797068045645


## 2. Решение систем линейных алгебраических уравнений

In [5]:
def gauss_elimination(A, b):
    # Метод Гаусса с выбором главного элемента
    n = len(b)
    A = A.copy().astype(float)
    b = b.copy().astype(float)
    for i in range(n):
        # Выбор главного элемента
        max_row = np.argmax(np.abs(A[i:, i])) + i
        A[[i, max_row]] = A[[max_row, i]]
        b[[i, max_row]] = b[[max_row, i]]
        for j in range(i+1, n):
            factor = A[j, i] / A[i, i]
            A[j, i:] -= factor * A[i, i:]
            b[j] -= factor * b[i]
    # Обратный ход
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
    return x

def jacobi_iteration(A, b, x0, tol=1e-6, max_iter=100):
    # Метод простых итераций (Якоби)
    n = len(b)
    x = x0.copy()
    for _ in range(max_iter):
        x_new = np.zeros(n)
        for i in range(n):
            x_new[i] = (b[i] - np.dot(A[i, :i], x[:i]) - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

# Пример
A = np.array([[4, -1, 0], [-1, 4, -1], [0, -1, 4]], dtype=float)
b = np.array([15, 10, 10], dtype=float)

x_gauss = gauss_elimination(A, b)
print("Решение Гауссом:", x_gauss)

x_jacobi = jacobi_iteration(A, b, np.zeros(3))
print("Решение Якоби:", x_jacobi)

# Сравнение: Гаусс точный, Якоби итерационный, подходит для больших разреженных матриц.

Решение Гауссом: [4.91071429 4.64285714 3.66071429]
Решение Якоби: [4.91071403 4.64285687 3.66071403]


## 3. Решение систем нелинейных уравнений

In [7]:
class NonlinearSystemSolver:
    def __init__(self, F, J=None):
        self.F = F  # F: vector function
        self.J = J  # Jacobian matrix

    def newton_system(self, x0, tol=1e-6, max_iter=100):
        # Метод Ньютона для систем
        if self.J is None:
            raise ValueError("Jacobian required")
        x = np.array(x0, dtype=float)
        for _ in range(max_iter):
            F_val = np.array(self.F(x))
            J_val = np.array(self.J(x))
            delta = np.linalg.solve(J_val, -F_val)
            x += delta
            if np.linalg.norm(delta) < tol:
                return x
        return x

    def fixed_point_system(self, G, x0, tol=1e-6, max_iter=100):
        # Метод простых итераций для систем
        x = np.array(x0, dtype=float)
        for _ in range(max_iter):
            x_new = np.array(G(x))
            if np.linalg.norm(x_new - x) < tol:
                return x_new
            x = x_new
        return x

    def modified_newton_system(self, x0, tol=1e-6, max_iter=100):
        # Модифицированный метод Ньютона (постоянная Якоби)
        if self.J is None:
            raise ValueError("Jacobian required")
        x = np.array(x0, dtype=float)
        J_inv = np.linalg.inv(np.array(self.J(x)))
        for _ in range(max_iter):
            F_val = np.array(self.F(x))
            delta = np.dot(J_inv, -F_val)
            x += delta
            if np.linalg.norm(delta) < tol:
                return x
        return x

# Пример: система {x^2 + y^2 = 1, x + y = 1}
def F(xy):
    x, y = xy
    return [x**2 + y**2 - 1, x + y - 1]

def J(xy):
    x, y = xy
    return [[2*x, 2*y], [1, 1]]

solver_sys = NonlinearSystemSolver(F, J)
root_newton_sys = solver_sys.newton_system([0.8, 0.2])
print("Решение системы Ньютона:", root_newton_sys)

Решение системы Ньютона: [ 1.00000000e+00 -5.41050765e-17]
